<a href="https://colab.research.google.com/github/lyudmildimitrov1997-sudo/pppp/blob/main/Case1_try_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [48]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

DATA MERGING & SUPPLIER ID VERIFICATION

In [49]:
def load_and_merge_data(file1_path, file2_path):
    df1 = pd.read_csv(file1_path)
    df2 = pd.read_csv(file2_path)

    print("--- Data Verification ---")
    print(f"File 1 Shape (Security Metrics): {df1.shape}")
    print(f"File 2 Shape (Business Context): {df2.shape}")

    ids1 = set(df1['supplier_id'])
    ids2 = set(df2['supplier_id'])

    if ids1 == ids2:
        print(" Verification Success: All supplier IDs match perfectly between datasets.\n")
    else:
        print(f" Warning: ID Mismatch detected! Unique IDs in File 1: {len(ids1)}, File 2: {len(ids2)}\n")

    merged_df = pd.merge(df1, df2, on='supplier_id', how='inner')
    return merged_df

EXPLORE MISSING VALUES & TARGET VARIABLE CLASS DISTRIBUTION

In [50]:
def explore_data(df):
    """
    Explores missing data points and evaluates target class imbalance.
    """
    print("--- Missing Values Exploration ---")
    missing_vals = df.isnull().sum()
    missing_with_counts = missing_vals[missing_vals > 0]
    if len(missing_with_counts) > 0:
        print(missing_with_counts)
    else:
        print("No missing values found in the dataset.")

    print("\n--- Target Variable Class Distribution ('cybersecurity_incident_past_12m') ---")
    print("Percentage Distribution:")
    print(df['cybersecurity_incident_past_12m'].value_counts(normalize=True) * 100)
    print("\nAbsolute Value Counts:")
    print(df['cybersecurity_incident_past_12m'].value_counts())
    print("\n" + "="*50 + "\n")

MACHINE LEARNING MODELS TRAINING & EVALUATION

In [51]:
def train_and_evaluate_models(df):
    """
    Splits features and target, processes data through pipelines (imputation,
    scaling, encoding), trains multiple models, and tracks performance metrics.
    """
    # Dropping high-cardinality/identifiers that won't help the predictive model
    X = df.drop(columns=['supplier_id', 'supplier_name', 'cybersecurity_incident_past_12m'])
    y = df['cybersecurity_incident_past_12m'].astype(int)

    # Automatically separate numerical and categorical attributes
    numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = X.select_dtypes(include=['object', 'bool']).columns.tolist()

    # Building processing sub-pipelines
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')), # Handles missing numerical metrics safely
        ('scaler', StandardScaler())                  # Centers data around a mean of 0 and unit variance
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')), # Fills categorical values via mode
        ('encoder', OneHotEncoder(handle_unknown='ignore'))   # Translates text/booleans to One-Hot columns
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ])

    # Split into Train (80%) and Test (20%) sets using stratification to keep stable class proportions
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Dictionary containing the candidate classification algorithms
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'Random Forest': RandomForestClassifier(random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(random_state=42),
        'Support Vector Machine (SVM)': SVC(probability=True, random_state=42)
    }

    metrics_summary = {}

    print("--- Model Performance Results ---")
    for name, model in models.items():
        # Coupling the specific preprocessing steps with the model inside a single pipeline
        pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                   ('classifier', model)])

        # Train model
        pipeline.fit(X_train, y_train)

        # Run predictions
        y_pred = pipeline.predict(X_test)
        y_prob = pipeline.predict_proba(X_test)[:, 1]

        # Calculate metric weights
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_prob)

        metrics_summary[name] = {'Accuracy': acc, 'F1-Score': f1, 'ROC-AUC': auc}

        print(f"\nModel Name: {name}")
        print(f"  Accuracy:  {acc:.4f}")
        print(f"  F1-Score:  {f1:.4f}")
        print(f"  ROC-AUC:   {auc:.4f}")

    return metrics_summary

MAIN RUNTIME

In [52]:
if __name__ == "__main__":
    # Define local CSV file paths matching your saved datasets
    file_security = "supplier_security_metrics.csv"
    file_business = "supplier_business_context.csv"

    try:
        # Execute the standard analytical pipeline steps
        df_merged = load_and_merge_data(file_security, file_business)
        explore_data(df_merged)
        results_summary = train_and_evaluate_models(df_merged)
    except FileNotFoundError:
        print("Execution Hint: Please save the provided raw text columns into separate local CSV files before running.")

Execution Hint: Please save the provided raw text columns into separate local CSV files before running.
